## Task 2: Gas Storage Contract Pricing
**Goal:** Build a function that prices a gas storage contract  
**Input:** 6 parameters (injection dates, withdrawal dates, prices, rate, max volume, storage costs)  
**Output:** Contract value in dollars  
**Formula:** Value = Revenue − Buy Cost − Storage Cost − Injection/Withdrawal Cost

In [ ]:
# --- Rebuild Task 1 dependencies ---
import pandas as pd
import numpy as np
from scipy.interpolate import CubicSpline

# load and clean
df = pd.read_csv("Nat_gas.csv")
df["Dates"] = pd.to_datetime(df["Dates"])
df["Days"] = (df["Dates"] - df["Dates"].min()).dt.days

# trend line (linear fit)
z = np.polyfit(df["Days"], df["Prices"], 1)
p = np.poly1d(z)

# seasonal correction
df["Trend"] = p(df["Days"])
df["Residual"] = df["Prices"] - df["Trend"]
seasonal_correction = df.groupby(df["Dates"].dt.month)["Residual"].mean()

# cubic spline (interpolation)
cs = CubicSpline(df["Days"], df["Prices"])

# the function from Task 1
def estimate_price(date):
    date = pd.to_datetime(date)
    day_number = (date - df["Dates"].min()).days
    min_day = df["Days"].min()
    max_day = df["Days"].max()
    
    if min_day <= day_number <= max_day:
        price = cs(day_number)
    else:
        trend_value = p(day_number)
        month = date.month
        correction = seasonal_correction[month]
        price = trend_value + correction
    
    return round(float(price), 2)

# quick test
print("Setup complete!")
print("Test - Oct 31, 2020:", estimate_price("2020-10-31"))
print("Test - Mar 31, 2025:", estimate_price("2025-03-31"))

### Step 1: Build and Test the Timeline Logic
- Combine injection and withdrawal dates into one sorted list
- Label each event as "inject" or "withdraw"
- Look up prices using estimate_price() from Task 1
- Walk through events in order, tracking cash flows

In [ ]:
# test the timeline logic separately
injection_dates = ["2025-06-01", "2025-07-01"]
withdrawal_dates = ["2025-12-01"]

events = []

for date in injection_dates:
    events.append({"date": pd.to_datetime(date), "type": "inject"})

for date in withdrawal_dates:
    events.append({"date": pd.to_datetime(date), "type": "withdraw"})

events.sort(key=lambda x: x["date"])

for e in events:
    print(e)

In [ ]:
# add prices from estimate_price
for e in events:
    e["price"] = estimate_price(e["date"])
    print(e["date"].date(), e["type"], "$", e["price"])

In [ ]:
# walk through each event and calculate cash flows
injection_rate = 1_000_000
withdrawal_rate = 1_000_000
max_volume = 1_000_000
storage_cost_per_day = 0.001
injection_withdrawal_cost_rate = 0.0005

total_revenue = 0
total_buy_cost = 0
total_storage_cost = 0
total_injection_withdrawal_cost = 0
current_volume = 0

for i, event in enumerate(events):
    
    # storage cost since last event
    if i > 0:
        days_between = (event["date"] - events[i-1]["date"]).days
        total_storage_cost += current_volume * storage_cost_per_day * days_between
    
    if event["type"] == "inject":
        volume = min(injection_rate, max_volume - current_volume)
        current_volume += volume
        total_buy_cost += volume * event["price"]
        total_injection_withdrawal_cost += volume * injection_withdrawal_cost_rate
        print(f"INJECT {volume:,} MMBtu on {event['date'].date()} at ${event['price']}")
    
    elif event["type"] == "withdraw":
        volume = min(withdrawal_rate, current_volume)
        current_volume -= volume
        total_revenue += volume * event["price"]
        total_injection_withdrawal_cost += volume * injection_withdrawal_cost_rate
        print(f"WITHDRAW {volume:,} MMBtu on {event['date'].date()} at ${event['price']}")

value = total_revenue - total_buy_cost - total_storage_cost - total_injection_withdrawal_cost

print(f"\nRevenue:                ${total_revenue:,.2f}")
print(f"Buy cost:              -${total_buy_cost:,.2f}")
print(f"Storage cost:          -${total_storage_cost:,.2f}")
print(f"Inject/withdraw cost:  -${total_injection_withdrawal_cost:,.2f}")
print(f"Contract value:         ${value:,.2f}")

### Step 2: The Pricing Function
Wraps the tested logic into a reusable function.

**Cash flows calculated:**
1. Revenue — volume × sell price on each withdrawal date
2. Buy cost — volume × buy price on each injection date
3. Storage cost — volume in tank × cost per day × days between events
4. Injection/withdrawal cost — volume pumped × cost rate per MMBtu

**Constraints enforced:**
- Cannot inject more than remaining tank capacity
- Cannot withdraw more than current volume in tank

In [ ]:
def price_contract(injection_dates, withdrawal_dates,
                   injection_rate, withdrawal_rate,
                   max_volume, storage_cost_per_day,
                   injection_withdrawal_cost_rate):
    
    # Step A: Build sorted timeline
    events = []
    for date in injection_dates:
        date = pd.to_datetime(date)
        events.append({"date": date, "type": "inject", "price": estimate_price(date)})
    for date in withdrawal_dates:
        date = pd.to_datetime(date)
        events.append({"date": date, "type": "withdraw", "price": estimate_price(date)})
    events.sort(key=lambda x: x["date"])
    
    # Step B: Walk through events, calculate cash flows
    total_revenue = 0
    total_buy_cost = 0
    total_storage_cost = 0
    total_iw_cost = 0
    current_volume = 0
    
    for i, event in enumerate(events):
        if i > 0:
            days_between = (event["date"] - events[i-1]["date"]).days
            total_storage_cost += current_volume * storage_cost_per_day * days_between
        
        if event["type"] == "inject":
            volume = min(injection_rate, max_volume - current_volume)
            current_volume += volume
            total_buy_cost += volume * event["price"]
            total_iw_cost += volume * injection_withdrawal_cost_rate
        
        elif event["type"] == "withdraw":
            volume = min(withdrawal_rate, current_volume)
            current_volume -= volume
            total_revenue += volume * event["price"]
            total_iw_cost += volume * injection_withdrawal_cost_rate
    
    # Step C: Contract value
    value = total_revenue - total_buy_cost - total_storage_cost - total_iw_cost
    return value

print("Function defined!")

### Step 3: Testing
- Test 1: Simple buy summer / sell winter (should be positive)
- Test 2: Multiple injections (500K + 500K)
- Test 3: Overfill protection (800K + 800K into 1M tank)
- Test 4: Bad trade — buy winter / sell summer (should be negative)

In [ ]:
# Test 1: Simple — one injection, one withdrawal
value = price_contract(
    injection_dates=["2025-06-01"],
    withdrawal_dates=["2025-12-01"],
    injection_rate=1_000_000,
    withdrawal_rate=1_000_000,
    max_volume=1_000_000,
    storage_cost_per_day=0.001,
    injection_withdrawal_cost_rate=0.0005
)

buy_price = estimate_price("2025-06-01")
sell_price = estimate_price("2025-12-01")
days_stored = (pd.to_datetime("2025-12-01") - pd.to_datetime("2025-06-01")).days

print(f"Buy price (Jun):   ${buy_price}/MMBtu")
print(f"Sell price (Dec):  ${sell_price}/MMBtu")
print(f"Days stored:       {days_stored}")
print(f"Contract value:    ${value:,.2f}")

In [ ]:
# Test 2: Multiple injections, one withdrawal
value2 = price_contract(
    injection_dates=["2025-06-01", "2025-07-01"],
    withdrawal_dates=["2025-12-01"],
    injection_rate=500_000,
    withdrawal_rate=1_000_000,
    max_volume=1_000_000,
    storage_cost_per_day=0.001,
    injection_withdrawal_cost_rate=0.0005
)
print(f"Test 2 - Two injections: ${value2:,.2f}")

# Test 3: Overfill protection
value3 = price_contract(
    injection_dates=["2025-06-01", "2025-07-01"],
    withdrawal_dates=["2025-12-01"],
    injection_rate=800_000,
    withdrawal_rate=1_000_000,
    max_volume=1_000_000,
    storage_cost_per_day=0.001,
    injection_withdrawal_cost_rate=0.0005
)
print(f"Test 3 - Overfill (800K+800K into 1M tank): ${value3:,.2f}")

# Test 4: Bad trade — buy winter, sell summer
value4 = price_contract(
    injection_dates=["2025-01-01"],
    withdrawal_dates=["2025-07-01"],
    injection_rate=1_000_000,
    withdrawal_rate=1_000_000,
    max_volume=1_000_000,
    storage_cost_per_day=0.001,
    injection_withdrawal_cost_rate=0.0005
)
print(f"Test 4 - Bad trade (buy winter, sell summer): ${value4:,.2f}")